# Symptom Embedding Space Visualization

This notebook inspects the quality of the symptom embeddings stored in Neo4j (see `generate-embeddings.ipynb`) by:

1. Fetching each symptom's embedding vector from the graph,
2. Grouping semantically similar symptoms with K-Means, choosing the number of clusters via silhouette score,
3. Projecting the high-dimensional embeddings to 2D with t-SNE,
4. Rendering an interactive scatter plot so clusters and outliers can be inspected by hovering over individual symptoms.

## Package Installation

In [ ]:
%pip install neo4j numpy scikit-learn plotly python-dotenv

## Imports and Configuration

In [ ]:
import os
from pathlib import Path

import numpy as np
import plotly.express as px
from dotenv import load_dotenv
from neo4j import GraphDatabase
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score

In [ ]:
QUERY_SYMPTOM_EMBEDDINGS = """
MATCH (s:Symptom)
WHERE s.embedding IS NOT NULL
RETURN s.n4sch__label[0] AS label, s.embedding AS embedding
"""

RANDOM_STATE = 42
MIN_CLUSTERS = 2
MAX_CLUSTERS_CAP = 10

RESULTS_DIR = Path("results") / "embeddings-visualization"
OUTPUT_FILENAME = "tsne_symptoms.html"

## Connecting to the Neo4j Graph Database

In [ ]:
load_dotenv()

neo4j_url = os.getenv("NEO4J_URL")
neo4j_username = os.getenv("NEO4J_USERNAME")
neo4j_password = os.getenv("NEO4J_PASSWORD")

if not all([neo4j_url, neo4j_username, neo4j_password]):
    raise RuntimeError("Missing Neo4j connection settings in environment variables.")

driver = GraphDatabase.driver(
    neo4j_url,
    auth=(neo4j_username, neo4j_password),
)

## Fetching Symptom Embeddings

The driver is closed as soon as the data is fetched, since no further database access is needed for the rest of the notebook.

In [ ]:
print("Fetching symptom embeddings from Neo4j...")

labels = []
embeddings = []

try:
    with driver.session() as session:
        result = session.run(QUERY_SYMPTOM_EMBEDDINGS)
        for record in result:
            label = record["label"]
            embedding = record["embedding"]

            if label and embedding:
                labels.append(label)
                embeddings.append(embedding)
finally:
    driver.close()

num_samples = len(labels)
print(f"Fetched {num_samples} symptoms with embeddings.")

In [ ]:
if num_samples < MIN_CLUSTERS + 1:
    raise RuntimeError(
        f"Need at least {MIN_CLUSTERS + 1} symptoms with embeddings to run this "
        f"analysis (found {num_samples})."
    )

embedding_lengths = {len(embedding) for embedding in embeddings}
if len(embedding_lengths) > 1:
    raise RuntimeError(f"Embeddings have inconsistent dimensions: {embedding_lengths}")

X = np.array(embeddings)
print(f"Embedding matrix shape: {X.shape}")

## Selecting the Number of Clusters (Silhouette Analysis)

Rather than picking a fixed cluster count, we search a range of candidate values for *k* and keep the one that maximizes the [silhouette score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html), which rewards clusters that are dense and well separated.

In [ ]:
max_clusters = min(MAX_CLUSTERS_CAP, num_samples - 1)
print(f"Searching k in [{MIN_CLUSTERS}, {max_clusters}] using silhouette score...")

silhouette_scores = {}
for k in range(MIN_CLUSTERS, max_clusters + 1):
    trial_kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init="auto")
    trial_cluster_ids = trial_kmeans.fit_predict(X)
    silhouette_scores[k] = silhouette_score(X, trial_cluster_ids)
    print(f"  k={k}: silhouette={silhouette_scores[k]:.4f}")

best_k = max(silhouette_scores, key=silhouette_scores.get)
print(f"\nBest k by silhouette score: {best_k} (score={silhouette_scores[best_k]:.4f})")

## Clustering with K-Means

In [ ]:
print(f"Running K-Means clustering with k={best_k}...")

kmeans = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init="auto")
cluster_ids = kmeans.fit_predict(X)

cluster_to_symptoms = {}
for label, cluster_id in zip(labels, cluster_ids):
    cluster_to_symptoms.setdefault(cluster_id, []).append(label)

cluster_names = {
    cluster_id: f"Cluster {cluster_id} ({', '.join(members[:2])})"
    for cluster_id, members in cluster_to_symptoms.items()
}

## Dimensionality Reduction with t-SNE

In [ ]:
perplexity = min(30, max(1, num_samples // 2))
print(f"Running t-SNE dimensionality reduction with perplexity={perplexity}...")

tsne = TSNE(
    n_components=2,
    perplexity=perplexity,
    random_state=RANDOM_STATE,
    init="pca",
    learning_rate="auto",
)
X_2d = tsne.fit_transform(X)

## Interactive Visualization

Each point is a symptom; hover over a point to see its label. The plot is saved as a standalone HTML file so it can be shared or reopened without rerunning the notebook.

In [ ]:
plot_clusters = [cluster_names[cluster_id] for cluster_id in cluster_ids]

fig = px.scatter(
    x=X_2d[:, 0],
    y=X_2d[:, 1],
    color=plot_clusters,
    hover_name=labels,
    labels={"x": "t-SNE Dimension 1", "y": "t-SNE Dimension 2", "color": "Cluster"},
    title="2D Projection of Symptom Vector Space (t-SNE + K-Means)",
    width=1000,
    height=700,
)
fig.update_traces(
    marker=dict(size=10, opacity=0.85, line=dict(width=0.5, color="white"))
)
fig.update_layout(legend_title_text="Symptom Clusters (Key Examples)")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
output_path = RESULTS_DIR / OUTPUT_FILENAME
fig.write_html(output_path)
print(f"Interactive plot saved to '{output_path}'.")

fig.show()